# Open Dataset

In [9]:
import pandas as pd

df = pd.read_csv('data/Customertravel.csv')
print(df)

     Age FrequentFlyer AnnualIncomeClass  ServicesOpted  \
0     34            No     Middle Income              6   
1     34           Yes        Low Income              5   
2     37            No     Middle Income              3   
3     30            No     Middle Income              2   
4     30            No        Low Income              1   
..   ...           ...               ...            ...   
949   31           Yes        Low Income              1   
950   30            No     Middle Income              5   
951   37            No     Middle Income              4   
952   30            No        Low Income              1   
953   31           Yes       High Income              1   

    AccountSyncedToSocialMedia BookedHotelOrNot  Target  
0                           No              Yes       0  
1                          Yes               No       1  
2                          Yes               No       0  
3                           No               No       0  
4

## Remove duplicadas

In [ ]:
total_duplicates = df.duplicated().sum()
print(f"Total de linhas duplicadas:  {total_duplicates}")

Total duplicates lines:  507


In [ ]:
total_lines = len(df)
total_duplicates = df.duplicated().sum()
percent = (total_duplicates / total_lines) * 100

print(f"Total de linhas: {total_lines}")
print(f"Linhas duplicadas: {total_duplicates}({percent:.2f}%)")

Total lines: 954
Duplicated lines: 507(53.14%)


In [14]:
df.nunique()

Age                           11
FrequentFlyer                  3
AnnualIncomeClass              3
ServicesOpted                  6
AccountSyncedToSocialMedia     2
BookedHotelOrNot               2
Target                         2
dtype: int64

In [16]:
duplicated = df[df.duplicated(keep=False)]
display(duplicated.sort_values(by=list(df.columns)))

,Age,FrequentFlyer,AnnualIncomeClass,ServicesOpted,AccountSyncedToSocialMedia,BookedHotelOrNot,Target
148,27,No,Low Income,1,No,No,0
184,27,No,Low Income,1,No,No,0
724,27,No,Low Income,1,No,No,0
46,27,No,Low Income,1,Yes,No,0
346,27,No,Low Income,1,Yes,No,0
...,...,...,...,...,...,...,...
567,38,No,Middle Income,2,Yes,No,0
241,38,Yes,Low Income,1,No,No,0
889,38,Yes,Low Income,1,No,No,0
330,38,Yes,Low Income,6,No,Yes,0


## Decisão final

Após análise do dataset, foi notado que as variáveis são quase todas binárias (Yes/No) ou categóricas com poucas opções. Existem
milhares de de combinações possíveis, porém com +900 linhas, é normal que existam dois clientes de 30 anos, de renda média, que não são _frequent flyers_ e usaram 1 serviço. Então a decisão final foi **MANTER** as duplicatas, pois representam clientes reais diferentes e a remoção causaria a perda de volume real de dados.

# Remove nulos

In [17]:
import numpy as np

df.replace("No Record", np.nan, inplace=True)

print("Nulos encontrados por coluna:")
print(df.isnull().sum())

df.dropna(inplace=True)

print("\nTamnho do dataset após remover nulos: ", df.shape)

Nulos encontrados por coluna:
Age                            0
FrequentFlyer                 60
AnnualIncomeClass              0
ServicesOpted                  0
AccountSyncedToSocialMedia     0
BookedHotelOrNot               0
Target                         0
dtype: int64

Tamnho do dataset após remover nulos:  (894, 7)


# Conversão de variáveis categóricas (Encoding)

In [18]:
from sklearn.preprocessing import LabelEncoder

# Mapeamento ordinal para AnnualIncomeClass
# Low = 0, Middle = 1, High = 2
df['AnnualIncomeClass'] = df['AnnualIncomeClass'].map({'Low Income': 0, 'Middle Income': 1, 'High Income': 2})

# Mapeamento binário para as outras colunas (One-Hot Encoding manual)
df['FrequentFlyer'] = df['FrequentFlyer'].map({'No': 0, 'Yes': 1})
df['AccountSyncedToSocialMedia'] = df['AccountSyncedToSocialMedia'].map({'No': 0, 'Yes': 1})
df['BookedHotelOrNot'] = df['BookedHotelOrNot'].map({'No': 0, 'Yes': 1})

display(df.head())

,Age,FrequentFlyer,AnnualIncomeClass,ServicesOpted,AccountSyncedToSocialMedia,BookedHotelOrNot,Target
0,34,0,1,6,0,1,0
1,34,1,0,5,1,0,1
2,37,0,1,3,1,0,0
3,30,0,1,2,0,0,0
4,30,0,0,1,0,0,0


# Separação de Variáveis e Train-Test Split (80/20)

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Target', axis=1)
y = df['Target']

# Divisão 80% treino e 20% teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste: {X_test.shape[0]} amostras')

Treino: 715 amostras
Teste: 179 amostras


# Normalização / Padronização (Scaling)

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

scaler = StandardScaler()

# Fit e transform no treino
X_train_scaled = scaler.fit_transform(X_train)

# Apenas transform no teste
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns)
display(X_train_scaled_df.head())

,Age,FrequentFlyer,AnnualIncomeClass,ServicesOpted,AccountSyncedToSocialMedia,BookedHotelOrNot
0,1.160601,-0.701921,0.311299,1.016436,-0.762796,1.268656
1,-0.331362,-0.701921,-1.037662,-0.250786,-0.762796,-0.788235
2,0.563816,-0.701921,-1.037662,-0.250786,-0.762796,1.268656
3,-0.629754,-0.701921,0.311299,1.016436,-0.762796,1.268656
4,1.757386,1.424662,-1.037662,0.382825,-0.762796,1.268656


# Exportar Dataset Limpo e Encodado
Para que possamos usá-lo no pipeline de modelos (	rain-models.ipynb).

In [ ]:
df.to_csv('data/cleaned_dataset.csv', index=False)
print('Dataset exportado com sucesso!')